# YOLOv8n 단독 학습

런타임 → 런타임 유형 변경 → T4 GPU

`train_compare_colab.ipynb`과 `QUICK_MODE`를 동일하게 둘 것. 결과는 `yolov8n_result.csv`로 나오며 비교표와 열 구성이 같다.

In [ ]:
%pip install -q ultralytics

import torch
assert torch.cuda.is_available(), "GPU 런타임 아님"
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

RUNS_DIR = Path('/content/runs')
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(RUNS_DIR)

In [ ]:
from google.colab import files

DATASET_ZIP = 'Pipe Crack Detection.v8-pipe-crack.yolov8.zip'

ZIP_PATH = next((p for p in (Path.cwd() / DATASET_ZIP, Path('/content') / DATASET_ZIP)
                 if p.is_file()), None)

if ZIP_PATH is None:
    print(f"{DATASET_ZIP} 선택")
    uploaded = files.upload()
    ZIP_PATH = Path('/content') / next(iter(uploaded))

print(ZIP_PATH, f"{ZIP_PATH.stat().st_size / 1e6:.1f} MB")

In [ ]:
import shutil, yaml

DATASET_DIR = Path('/content/dataset')
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
DATASET_DIR.mkdir(parents=True)
shutil.unpack_archive(str(ZIP_PATH), str(DATASET_DIR))

DATA_YAML = DATASET_DIR / 'data.yaml'
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg.pop('path', None)
for key, split in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
    images = DATASET_DIR / split / 'images'
    if images.is_dir():
        cfg[key] = str(images)
    else:
        cfg.pop(key, None)
DATA_YAML.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

CLASS_NAMES = cfg['names']
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[k] for k in sorted(CLASS_NAMES)]
CRACK_IDX = CLASS_NAMES.index('Crack') if 'Crack' in CLASS_NAMES else 0

for split in ('train', 'valid', 'test'):
    print(split, len(list((DATASET_DIR / split / 'images').glob('*'))))
print(CLASS_NAMES, '| Crack idx', CRACK_IDX)

In [ ]:
TAG = 'yolov8n'
WEIGHTS = 'yolov8n.pt'

QUICK_MODE = True

TRAIN_ARGS = dict(
    data=str(DATA_YAML),
    epochs=25 if QUICK_MODE else 60,
    patience=10 if QUICK_MODE else 20,
    imgsz=640,
    batch=16,
    seed=0,
    mosaic=0.5,
    close_mosaic=10,
    project=str(RUNS_DIR),
    exist_ok=True,
    plots=True,
    verbose=False,
)

TARGET_HZ = 10.0
LATENCY_BUDGET_MS = 1000.0 / TARGET_HZ

print(TAG, '| epochs', TRAIN_ARGS['epochs'], '| budget', LATENCY_BUDGET_MS, 'ms')

In [ ]:
import time, gc, json
from ultralytics import YOLO

started = time.time()
model = YOLO(WEIGHTS)
result = model.train(name=TAG, **TRAIN_ARGS)
TRAIN_SEC = time.time() - started

SAVE_DIR = Path(result.save_dir)
BEST = SAVE_DIR / 'weights' / 'best.pt'
print(f"{TRAIN_SEC / 60:.1f}분  {BEST}")

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import numpy as np
import pandas as pd

test_images = [str(p) for p in sorted((DATASET_DIR / 'test' / 'images').glob('*.jpg'))]
LATENCY_SAMPLES = test_images[:60]

model = YOLO(str(BEST))

try:
    _, n_params, _, gflops = model.info(verbose=False)
except Exception:
    n_params, gflops = float('nan'), float('nan')

m = model.val(data=str(DATA_YAML), split='test', imgsz=640, verbose=False)
try:
    pos = list(m.box.ap_class_index).index(CRACK_IDX)
    crack_map50, crack_map = float(m.box.ap50[pos]), float(m.box.ap[pos])
except (ValueError, IndexError):
    crack_map50 = crack_map = float('nan')

model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
preds = model.predict(source=LATENCY_SAMPLES, imgsz=640, verbose=False)
s = preds[0].speed
latency = s['preprocess'] + s['inference'] + s['postprocess']

cc = [p.boxes.conf[p.boxes.cls == CRACK_IDX].cpu().numpy() for p in preds]
cc = np.concatenate([c for c in cc if len(c)]) if any(len(c) for c in cc) else np.array([])

row = {
    '모델': TAG,
    'Crack mAP50': round(crack_map50, 4),
    'Crack mAP50-95': round(crack_map, 4),
    '전체 mAP50': round(float(m.box.map50), 4),
    '전체 mAP50-95': round(float(m.box.map), 4),
    'P': round(float(m.box.mp), 4),
    'R': round(float(m.box.mr), 4),
    '지연(ms)': round(latency, 1),
    '파라미터(M)': round(n_params / 1e6, 2),
    'GFLOPs': round(gflops, 1),
    '학습(분)': round(TRAIN_SEC / 60, 1),
    'Crack conf>=0.8': int((cc >= 0.8).sum()),
    'Crack 검출수': int(len(cc)),
    'epochs': TRAIN_ARGS['epochs'],
}
row['10Hz'] = row['지연(ms)'] <= LATENCY_BUDGET_MS

df = pd.DataFrame([row])
df.to_csv(RUNS_DIR / 'yolov8n_result.csv', index=False)

del model
gc.collect()
torch.cuda.empty_cache()
df

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

for name in ('results.png', 'confusion_matrix_normalized.png', 'PR_curve.png'):
    p = SAVE_DIR / name
    if p.exists():
        print(name)
        display(IPyImage(filename=str(p), width=900))

In [ ]:
model = YOLO(str(BEST))
preds = model.predict(source=test_images[:6], imgsz=640, conf=0.25, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(13, 9))
for ax, pred in zip(axes.ravel(), preds):
    ax.imshow(pred.plot()[:, :, ::-1])
    ax.axis('off')
plt.tight_layout()
plt.show()

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
OUT = Path('/content/yolov8n_out')
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()

df.to_csv(OUT / 'yolov8n_result.csv', index=False)
shutil.copy(BEST, OUT / 'yolov8n_best.pt')
for f in ('results.png', 'results.csv', 'confusion_matrix_normalized.png', 'PR_curve.png'):
    if (SAVE_DIR / f).exists():
        shutil.copy(SAVE_DIR / f, OUT / f'yolov8n_{f}')

archive = shutil.make_archive('/content/yolov8n_out', 'zip', str(OUT))
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
print(sorted(p.name for p in OUT.iterdir()))

files.download(archive)